# Suite RRN — Reruns and diffs: queries over the certificates

A rerun reuses the parameter assignments an earlier history recorded, and a diff compares two histories
([CAP-076: reruns reuse decisions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-076.md)). Neither needs code of mbse-transforms': a transform certificate records its transform
(`transform`, a `References.OfObject` value), its parameter assignments (`assignments`) and its sub-certificates, each
holding the frame its argument's visitor bore (`frame`, a `References.OfFrame` value), and certificates are
mbse-schemas data. So a rerun's reuse is a query over the certificates' store: the transform certificate, in the
earlier history, of the same transform on the same frame, and the parameter assignments it recorded. A diff is the same
query over two histories, the certificates it pairs compared by mbse-schemas' `Comparison`: added, removed (the
orphans), and changed (other parameter assignments recorded). The cases use `Transforms.Session`, `Transforms.apply`
and `Transforms.certify` as any caller does, and mbse-schemas' transactions, `Plain` and `Comparison` for the rest; no
call of mbse-transforms' reads a certificate.

The tests play the caller. The fixture is suite XF's shelf: a singleton `shelf.Shelf` owning `shelf.Item`s by key,
each with a `name` and a `label`; the pattern `shelf.Unlabelled`, an item without a label, declaring the parameter
`case` (a str, left open by a session); and the transform `Label`, whose procedure labels the sync's item in the case
its parameter assignment gives. The transforms are defined in a store of their own, `transforms`, and the certificates
written in another, `certs`. A frame binds each symbol by object reference: a store remade with the same owners and
keys holds the same frames, and a history recorded over one store is queried by the frames of another.

In [ ]:
import json

from mbse.Schemas.Framework import Builders as B, Comparison, Parametrics, Plain, Proxies, References
from mbse.Transforms import Transforms
from support import bear, resolve


def shelf(txn):
    """shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name and a label."""
    S = txn.Schemas
    return txn.uid("rrn.shelf/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Shelf": S.OfObject, "Item": S.OfObject, "Holds": S.OfRelation})
        .spec([
            lambda _: _.Shelf.name("shelf.Shelf").ref(True).singleton("shelf.TheShelf").adjacencies({
                "items": lambda t: t.relation([_.Holds]).me("shelf")}),
            lambda _: _.Item.name("shelf.Item").ref(True).properties({"name": S.OfStr, "label": S.OfStr}).adjacencies([
                lambda t: t.name("shelf").relation([_.Holds]).me("item").owner(True)]),
            lambda _: _.Holds.name("shelf.Holds").links([lambda t: t.name("shelf"), lambda t: t.name("item")])
            .properties({"key": S.OfStr}).uniques([["shelf", "key"], ["item"]]),
        ])]))])


def patterns(txn):
    """shelf.Unlabelled, an item without a label, declaring the parameter case."""
    S = txn.Schemas
    return txn.uid("rrn.patterns/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Unlabelled": S.OfPattern})
        .spec([
            lambda _: _.Unlabelled.name("shelf.Unlabelled").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.label(None)])
            .parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "case": lambda d: d.parameter(lambda p: p.description("upper or lower").for_value([S.OfStr]))}))]),
        ])]))])


def model(*names):
    """A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it."""
    store = B.Store()
    store.transaction(Parametrics.install)
    store.transaction(shelf)
    store.transaction(patterns)
    store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"s": t.shelf.Shelf})]))]))
    for name in names:
        with store.transaction():
            store.shelf.Shelf(store.singleton("shelf.TheShelf"), lambda s, n=name: s.items({
                n: lambda e: e.item(lambda i: i.name(n))}))
    return store


def items(store):
    """Each item on the shelf: its key, name and label."""
    with store.transaction():
        return [(e.key.get(), e.item.name.get(), e.item.label.get()) for e in store.singleton("shelf.TheShelf").items]


def carried(own, x, store):
    """The reference of `x` (an object, or a frame's bindings), made in the store `own`, as a value of `store`'s:
    written as plain data and read back (mbse-schemas' serialization)."""
    reference = References.of(own, x)
    return Plain.FromPlain(store)(store.registered(reference.as_schema().name),
                                  Plain.ToPlain(own)(reference.as_schema(), reference))


def recorded(store, history, transform, frame):
    """The query a rerun and a diff make: the frames, in the certificates' store, binding each transform certificate
    `c` in the history named `history` of the transform `transform` (a `References.OfObject` value) and its
    sub-certificate `f` holding the frame `frame` (a `References.OfFrame` value), matched by equality in the spec."""
    T = store.Transforms
    return store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"d": T.Histories, "h": T.HistoryCertificate, "c": T.TransformCertificate, "f": T.FrameCertificate})
        .spec([
            lambda _: _.d.histories({history: lambda e: e.history(_.h)}),
            lambda _: _.h.steps([lambda e: e.step(_.c)]),
            lambda _: _.c.transform(transform).frames([lambda e: e.frame(_.f)]),
            lambda _: _.f.frame(frame),
        ])]))]))


def certified(store, history):
    """The query a diff starts from: the frames binding each transform certificate `c` in the history named
    `history` and each of its sub-certificates `f`."""
    T = store.Transforms
    return store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols({"d": T.Histories, "h": T.HistoryCertificate, "c": T.TransformCertificate, "f": T.FrameCertificate})
        .spec([
            lambda _: _.d.histories({history: lambda e: e.history(_.h)}),
            lambda _: _.h.steps([lambda e: e.step(_.c)]),
            lambda _: _.c.frames([lambda e: e.frame(_.f)]),
        ])]))]))


def held(store, member):
    """The value object a certificate's property holds, as a value of the store's, outside any transaction: written
    as plain data and read back."""
    found = []
    member.as_object(lambda o: found.append(Plain.FromPlain(store)(o.as_schema(), Plain.ToPlain(store)(o.as_schema(), o))) or o)
    return found[0]


def assignments_of(certificate):
    """The parameter assignments a transform certificate records, read through the visitor protocol."""
    found = {}

    def each(entries):
        for name, value in entries:
            value.as_scalar(lambda s, name=name: found.__setitem__(name, s.get()) or s)
        return entries

    certificate.assignments.as_value(lambda a: a.as_container(lambda c: c.as_entries(each) or c) or a)
    return found


def same(a, b):
    """Whether two certificates, each read within the store's transaction, are equal as values of their schema
    (mbse-schemas' `Comparison`): a transform certificate's transform, prototype and parameter assignments."""
    return Comparison.OfObject(a.as_schema(), a).compare(Comparison.OfObject(b.as_schema(), b)) == 0


def key_of(store, frame):
    """The key on the shelf of the item a sub-certificate's frame binds, read from it as plain data."""
    found = []
    frame.as_object(lambda o: found.append(Plain.ToPlain(store)(o.as_schema(), o)) or o)
    return found[0]["binds"]["i"]["path"]["next"]["entry"]["properties"]["key"]


def shown(store, history):
    """Each transform certificate of a history: the key of the item its sub-certificate's frame binds, its
    parameter assignments, and who decided them."""
    with store.transaction():
        held_history = store.singleton("Transforms.TheHistories").histories.at(history).history
        return [(key_of(store, next(iter(e.step.frames)).frame.frame), assignments_of(e.step), str(d.by))
                for e, d in zip(held_history.steps, held_history.decisions)]


def run(store, data, history, decide):
    """The caller's session over the data's frames of shelf.Unlabelled: each candidate of Label, its open parameter
    answered by `decide(candidate)`, which gives the parameter assignments and who decided them; then applied, with the
    candidate's frame as the sync, and certified into the history named `history` in the certificates' store `store`."""
    frames = data.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
        data.registered("shelf.Unlabelled")]))]))
    with transforms.transaction(), data.transaction():
        Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform
        for (candidate,) in Transforms.Session([Label], (bear(data, "shelf.Unlabelled", f) for f in frames)):
            assignments, by = decide(candidate)
            taken = candidate.answer(**assignments)
            Transforms.apply(Label, taken.assignments, procedures, sync=taken.frame)
            Transforms.certify(store, Label, taken.assignments, history, by, sync=taken.frame)


def chosen(choices):
    """The caller deciding each item's case itself, by the item's name."""
    return lambda candidate: ({"case": choices[str(candidate.frame.i.name)]}, "caller")


transforms = model()  # the transforms' store: the shelf's schemas and patterns, no data, and the transforms
transforms.transaction(Transforms.install)
transforms.transaction(lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled))
transforms.transaction(lambda t: Transforms.define(t, "Copy", "label", t.shelf.Unlabelled))
certs = B.Store()  # the certificates' store, as the caller gives it
certs.transaction(Transforms.install)
procedures = Transforms.Procedures()


def label(source, sync, target, assignments):
    """Labels the sync's item with its name, in the case its parameter assignment gives."""
    name = str(sync.i.name)
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.label(name.upper() if assignments["case"] == "upper" else name.lower()))


procedures.register("label", label)
with transforms.transaction():
    label_ref, copy_ref = (carried(transforms, transforms.singleton("Transforms.TheCatalog").transforms.at(n).transform, certs)
                   for n in ("Label", "Copy"))  # each transform's reference, as a value of the certificates' store

<a id="RRN-01"></a>
## RRN-01 · A rerun reuses: for each frame of a fresh session, a query over the certificates' store finds the transform certificate, in the earlier history, of the same transform on the same frame, and the caller applies and certifies with the parameter assignments it recorded, decided "reused"

In [ ]:
first = model("bolt", "nut")
run(certs, first, "first", chosen({"bolt": "upper", "nut": "lower"}))
assert items(first) == [("bolt", "bolt", "BOLT"), ("nut", "nut", "nut")]


def reusing(store, history, data, otherwise):
    """The caller reusing what the history named `history` recorded: for a candidate, the query for the transform
    certificate of Label on the candidate's frame (its bindings, as certify records them, made in the data's store);
    where it finds one, the parameter assignments it recorded, decided "reused"; where none, `otherwise`'s decision."""
    def decide(candidate):
        found = recorded(store, history, label_ref, carried(data, {"i": candidate.frame.i}, store))
        if not found:  # a frame the earlier history never saw
            return otherwise(candidate)
        (given,) = found
        with store.transaction():  # read within a transaction: the certificate the frame given back binds
            return assignments_of(resolve(store, given["c"])), "reused"
    return decide


again = model("bolt", "nut", "washer")  # remade: the same owners and keys, and an item the first run never saw
run(certs, again, "second", reusing(certs, "first", again, chosen({"washer": "upper"})))
assert items(again) == [("bolt", "bolt", "BOLT"), ("nut", "nut", "nut"), ("washer", "washer", "WASHER")]
assert shown(certs, "first") == [("bolt", {"case": "upper"}, "caller"), ("nut", {"case": "lower"}, "caller")]
assert shown(certs, "second") == [
    ("bolt", {"case": "upper"}, "reused"), ("nut", {"case": "lower"}, "reused"), ("washer", {"case": "upper"}, "caller")]
with certs.transaction():
    histories = certs.singleton("Transforms.TheHistories").histories
    one, two = (histories.at(h).history.steps for h in ("first", "second"))
    assert same(one.at(0).step, two.at(0).step) and same(one.at(1).step, two.at(1).step)  # transform, prototype, assignments
    assert not same(one.at(0).step, two.at(1).step)  # bolt's and nut's: other assignments

<a id="RRN-02"></a>
## RRN-02 · A diff: two histories over stores that differ, by the same query and mbse-schemas' `Comparison` alone: added, removed (the orphans), and changed (other parameter assignments recorded)

In [ ]:
before, after = model("bolt", "nut", "washer"), model("bolt", "nut", "screw")  # washer removed, screw added
run(certs, before, "before", chosen({"bolt": "upper", "nut": "upper", "washer": "lower"}))
run(certs, after, "after", chosen({"bolt": "upper", "nut": "lower", "screw": "lower"}))  # nut decided differently


def steps_of(store, history):
    """Each transform certificate of a history, found by the query `certified`: its reference, and its transform and
    its sub-certificate's frame, as values of the store's, to query another history by."""
    found = certified(store, history)
    with store.transaction():
        return [(given["c"], held(store, resolve(store, given["c"]).transform), held(store, resolve(store, given["f"]).frame))
                for given in found]


def diff(store, a, b):
    """What the history named `b` removes from the one named `a` (the orphans: a certificate of a's with none in b's
    of the same transform on the same frame), what it adds (one of b's with none in a's), and what it changes (a pair
    whose certificates are not equal as values), each by reference."""
    ours, theirs = steps_of(store, a), steps_of(store, b)
    removed = [c for c, transform, frame in ours if not recorded(store, b, transform, frame)]
    added = [c for c, transform, frame in theirs if not recorded(store, a, transform, frame)]
    pairs = [(c, given["c"]) for c, transform, frame in ours for given in recorded(store, b, transform, frame)]
    with store.transaction():
        changed = [(c, d) for c, d in pairs if not same(resolve(store, c), resolve(store, d))]
    return removed, added, changed


def told(store, references):
    """Each transform certificate a reference designates: the key of the item its frame binds, and its parameter
    assignments."""
    with store.transaction():
        return [(key_of(store, next(iter(c.frames)).frame.frame), assignments_of(c))
                for c in (resolve(store, reference) for reference in references)]


removed, added, changed = diff(certs, "before", "after")
assert told(certs, removed) == [("washer", {"case": "lower"})]  # the orphan
assert told(certs, added) == [("screw", {"case": "lower"})]
assert [told(certs, pair) for pair in changed] == [[("nut", {"case": "upper"}), ("nut", {"case": "lower"})]]  # bolt: unchanged
removed, added, changed = diff(certs, "after", "before")  # the other way: added and removed swap
assert (told(certs, removed), told(certs, added)) == ([("screw", {"case": "lower"})], [("washer", {"case": "lower"})])
assert diff(certs, "first", "second")[0] == [] and diff(certs, "first", "second")[2] == []  # a rerun that reused

<a id="RRN-03"></a>
## RRN-03 · What a query can match: a sub-certificate by its recorded frame, a `References.OfFrame` value, by equality in a pattern spec, as a transform certificate by its transform, a `References.OfObject` value, by its parameter assignments, and by its sub-certificate's argument; equality is of the whole value

In [ ]:
T = certs.Transforms


def count(symbols, spec):
    """How many frames a map over the certificates' store gives, of the symbols and the spec given."""
    return len(certs.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
        .symbols(symbols).spec(spec)]))])))


nut = model("nut")
(given,) = nut.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([nut.registered("shelf.Unlabelled")]))]))
(witness,) = nut.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"i": t.shelf.Item}).spec([lambda _: _.i.name("nut")])]))]))
bound = carried(nut, given, certs)  # the frame's bindings: what certify records, from a store of its own
with nut.transaction():
    renamed = carried(nut, {"j": resolve(nut, given["i"])}, certs)  # the same item, bound to another symbol
assert count({"f": T.FrameCertificate}, [lambda _: _.f.frame(bound)]) == 4  # nut's, in first, second, before, after
assert count({"f": T.FrameCertificate}, [lambda _: _.f.frame(renamed)]) == 0  # another frame
assert "assigns" in Plain.ToPlain(nut)(References.of(nut, witness).as_schema(), References.of(nut, witness))
assert count({"f": T.FrameCertificate}, [lambda _: _.f.frame(carried(nut, witness, certs))]) == 0  # a witness: its values too
assert count({"c": T.TransformCertificate}, [lambda _: _.c.transform(label_ref)]) == 11  # every certificate so far
assert count({"c": T.TransformCertificate}, [lambda _: _.c.transform(copy_ref)]) == 0  # never applied
for argument, expected in (("sync", 4), ("source", 0)):  # the sub-certificate by its argument, the entry's key
    assert count({"c": T.TransformCertificate, "f": T.FrameCertificate}, [
        lambda _: _.c.frames({argument: lambda e: e.frame(_.f)}), lambda _: _.f.frame(bound)]) == expected
lower = [lambda _: _.d.histories({"after": lambda e: e.history(_.h)}), lambda _: _.h.steps([lambda e: e.step(_.c)]),
         lambda _: _.c.assignments({"case": "lower"})]  # the parameter assignments, a container, by equality
assert count({"d": T.Histories, "h": T.HistoryCertificate, "c": T.TransformCertificate}, lower) == 2  # nut and screw
assert len(recorded(certs, "first", label_ref, bound)) == 1 and recorded(certs, "first", copy_ref, bound) == []

<a id="RRN-04"></a>
## RRN-04 · Determinism: the same frames and the same choices give byte-identical certificates, as plain data, in two stores

In [ ]:
def written(choices):
    """The certificates of a run over a store of bolt and nut, deciding by `choices`, in a certificates' store of
    their own, as plain data (mbse-schemas' snapshot), serialized."""
    store = B.Store()
    store.transaction(Transforms.install)
    run(store, model("bolt", "nut"), "run", chosen(choices))
    return json.dumps(Plain.ToPlain(store).Snapshot())


once = written({"bolt": "upper", "nut": "lower"})
assert written({"bolt": "upper", "nut": "lower"}) == once  # the same frames and choices: the same bytes
assert written({"bolt": "upper", "nut": "upper"}) != once  # other choices: other certificates